# Activation Patching Pipeline for useEffect Dependency Circuit Analysis

Run all 4 experiments over N counterfactual pairs and generate heatmaps.

### Usage
    1. Set MODEL_NAME and DATASET_PATH below.
    2. Run this script in a Jupyter notebook or as a Python file.
    3. Heatmaps are saved as PNG files in OUTPUT_DIR.

### Requirements
    pip install transformer-lens plotly kaleido


In [15]:
%pip install git+https://github.com/TransformerLensOrg/TransformerLens
%pip install huggingface_hub==1.11.0
%pip install kaleido

  Cloning https://github.com/TransformerLensOrg/TransformerLens to /tmp/pip-req-build-cu8xg8vc
  Running command git clone --filter=blob:none --quiet https://github.com/TransformerLensOrg/TransformerLens /tmp/pip-req-build-cu8xg8vc
  Resolved https://github.com/TransformerLensOrg/TransformerLens to commit 5f7b02e83d33a27f0cda14fc814907d88e07418b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [16]:
import json
import torch
import numpy as np
import plotly.express as px
import plotly.io as pio
import transformer_lens.patching as patching
from transformer_lens.model_bridge import TransformerBridge
from pathlib import Path
from huggingface_hub import login

## Configuration

In [17]:
MODEL_NAME = "google/gemma-3-1b-pt"  # Change to your model
DATASET_PATH = "pilot_dataset.json"
OUTPUT_DIR = Path("patching_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# Only use substitution type (removal type may have token length mismatch)
USE_TYPES = ["substitution"]  # Add "removal" after verifying token lengths

## Setup

In [18]:
login()

In [19]:
torch.set_grad_enabled(False)

print(f"Loading model: {MODEL_NAME}")
model = TransformerBridge.boot_transformers(MODEL_NAME)
# model.enable_compatibility_mode()  # Uncomment if memory allows

Loading model: google/gemma-3-1b-pt


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [20]:
with open(DATASET_PATH) as f:
    dataset = json.load(f)

# Filter by type
dataset = [item for item in dataset if item["type"] in USE_TYPES]
N = len(dataset)
print(f"Dataset: {N} sets ({USE_TYPES})")

Dataset: 5 sets (['substitution'])


## Helpers

In [21]:
def find_measurement_pos(model, code_string):
    str_tokens = model.to_str_tokens(code_string)
    for i, tok in enumerate(str_tokens):
        if tok.strip() == '[' and i > 0 and '}' in str_tokens[i - 1]:
            return i + 1
    raise ValueError(f"Could not find measurement position in: {code_string[:100]}...")

In [24]:
def save_heatmap(data, title, filename, xaxis="Head", yaxis="Layer",
                 x_labels=None, y_labels=None):
    if isinstance(data, torch.Tensor):
        data = data.cpu().numpy()
    fig = px.imshow(
        data,
        labels=dict(x=xaxis, y=yaxis, color="Patching Effect"),
        x=x_labels,
        y=y_labels,
        title=title,
        color_continuous_midpoint=0,
        color_continuous_scale="RdBu",
    )
    fig.update_layout(width=max(800, data.shape[1] * 15), height=max(500, data.shape[0] * 25))
    filepath = OUTPUT_DIR / filename
    fig.write_image(str(filepath))
    print(f"  Saved: {filepath}")
    return fig


## Main Pipeline

In [25]:
# Storage for aggregatable results (experiments 2 & 4)
all_attn_head_results = []       # Experiment 2
all_every_head_results = []      # Experiment 4

In [27]:
for idx, item in enumerate(dataset):
    print(f"\n{'='*60}")
    print(f"Processing {item['id']} ({idx+1}/{N}): {item['description']}")
    print(f"{'='*60}")

    # --- Tokenize ---
    clean_tokens = model.to_tokens(item["clean"])
    corrupted_tokens = model.to_tokens(item["corrupted"])

    if clean_tokens.shape != corrupted_tokens.shape:
        print(f"  SKIPPING: token length mismatch "
              f"(clean={clean_tokens.shape[1]}, corrupted={corrupted_tokens.shape[1]})")
        continue

    measurement_pos = find_measurement_pos(model, item["clean"])
    clean_target_id = model.to_single_token(item["clean_target"])
    corrupted_target_id = model.to_single_token(item["corrupted_target"])

    print(f"  Measurement position: {measurement_pos}")
    print(f"  Clean target: '{item['clean_target']}' (id={clean_target_id})")
    print(f"  Corrupted target: '{item['corrupted_target']}' (id={corrupted_target_id})")

    # --- Logit diff function (closed over this set's parameters) ---
    _mp = measurement_pos
    _ct = clean_target_id
    _crt = corrupted_target_id

    def get_logit_diff(logits, pos=_mp, ct=_ct, crt=_crt):
        pos_logits = logits[0, pos, :]
        return pos_logits[ct] - pos_logits[crt]

    # --- Forward passes ---
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)
    corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)

    clean_baseline = get_logit_diff(clean_logits).item()
    corrupted_baseline = get_logit_diff(corrupted_logits).item()
    print(f"  Clean logit diff: {clean_baseline:.4f}")
    print(f"  Corrupted logit diff: {corrupted_baseline:.4f}")

    if abs(clean_baseline - corrupted_baseline) < 0.01:
        print(f"  SKIPPING: logit diff too small to be meaningful")
        continue

    # --- Metric function ---
    _cb = corrupted_baseline
    _clb = clean_baseline

    def useeffect_metric(logits, cb=_cb, clb=_clb):
        return (get_logit_diff(logits) - cb) / (clb - cb)

    # --- Experiment 1: Residual stream patching (per-set heatmap) ---
    print("  Running Experiment 1: resid_pre patching...")
    resid_results = patching.get_act_patch_resid_pre(
        model, corrupted_tokens, clean_cache, useeffect_metric
    )
    str_tokens = model.to_str_tokens(clean_tokens[0])
    x_labels = [f"{tok} {i}" for i, tok in enumerate(str_tokens)]
    save_heatmap(
        resid_results, f"Exp1: resid_pre — {item['id']}",
        f"exp1_resid_pre_{item['id']}.png",
        xaxis="Position", yaxis="Layer", x_labels=x_labels
    )

    # --- Experiment 2: Attention head output patching (aggregatable) ---
    print("  Running Experiment 2: attn_head_out patching (all pos)...")
    attn_head_results = patching.get_act_patch_attn_head_out_all_pos(
        model, corrupted_tokens, clean_cache, useeffect_metric
    )
    all_attn_head_results.append(attn_head_results)
    save_heatmap(
        attn_head_results, f"Exp2: attn_head_out — {item['id']}",
        f"exp2_attn_head_{item['id']}.png",
    )

    # --- Experiment 3: Block-level patching (per-set heatmaps) ---
    print("  Running Experiment 3: block_every patching...")
    block_results = patching.get_act_patch_block_every(
        model, corrupted_tokens, clean_cache, useeffect_metric
    )
    block_labels = ["Residual Stream", "Attn Output", "MLP Output"]
    for b_idx, label in enumerate(block_labels):
        save_heatmap(
            block_results[b_idx],
            f"Exp3: {label} — {item['id']}",
            f"exp3_block_{label.replace(' ', '_').lower()}_{item['id']}.png",
            xaxis="Position", yaxis="Layer", x_labels=x_labels
        )

    # --- Experiment 4: Per-head every component patching (aggregatable) ---
    print("  Running Experiment 4: attn_head_all_pos_every patching...")
    every_head_results = patching.get_act_patch_attn_head_all_pos_every(
        model, corrupted_tokens, clean_cache, useeffect_metric
    )
    all_every_head_results.append(every_head_results)
    component_labels = ["Output", "Query", "Key", "Value", "Pattern"]
    for c_idx, label in enumerate(component_labels):
        save_heatmap(
            every_head_results[c_idx],
            f"Exp4: {label} — {item['id']}",
            f"exp4_head_{label.lower()}_{item['id']}.png",
        )

    # Clean up cache to free memory
    del clean_cache, corrupted_cache, clean_logits, corrupted_logits
    torch.cuda.empty_cache() if torch.cuda.is_available() else None


Processing sub_01 (1/5): fetch user by id — userId substituted with theme
  Measurement position: 58
  Clean target: 'name' (id=1201)
  Corrupted target: 'theme' (id=14818)
  Clean logit diff: 7.9033
  Corrupted logit diff: 7.1474
  Running Experiment 1: resid_pre patching...


  0%|          | 0/1560 [00:00<?, ?it/s]

  Saved: patching_results/exp1_resid_pre_sub_01.png
  Running Experiment 2: attn_head_out patching (all pos)...


  0%|          | 0/104 [00:00<?, ?it/s]

  Saved: patching_results/exp2_attn_head_sub_01.png
  Running Experiment 3: block_every patching...


...

  0%|          | 0/104 [00:00<?, ?it/s]

  0%|          | 0/26 [00:00<?, ?it/s]

  0%|          | 0/26 [00:00<?, ?it/s]

  0%|          | 0/104 [00:00<?, ?it/s]

  Saved: patching_results/exp4_head_output_sub_05.png
  Saved: patching_results/exp4_head_query_sub_05.png
  Saved: patching_results/exp4_head_key_sub_05.png
  Saved: patching_results/exp4_head_value_sub_05.png
  Saved: patching_results/exp4_head_pattern_sub_05.png


In [28]:
if len(all_attn_head_results) > 0:
    print(f"\n{'='*60}")
    print(f"Aggregate results over {len(all_attn_head_results)} sets")
    print(f"{'='*60}")

    # --- Experiment 2: Average ---
    avg_attn = torch.stack(all_attn_head_results).mean(dim=0)
    save_heatmap(
        avg_attn,
        f"Exp2 Average: attn_head_out ({len(all_attn_head_results)} sets)",
        "exp2_attn_head_AVERAGE.png",
    )
    # Print top heads
    n_heads = avg_attn.shape[1]
    flat = avg_attn.flatten()
    top_indices = flat.abs().topk(min(10, flat.numel())).indices
    print("\n  Exp2 — Top 10 heads (averaged):")
    for rank, i in enumerate(top_indices):
        layer = i.item() // n_heads
        head = i.item() % n_heads
        print(f"    #{rank+1}: L{layer}H{head} = {avg_attn[layer, head]:+.4f}")

    # --- Experiment 4: Average ---
    avg_every = torch.stack(all_every_head_results).mean(dim=0)
    component_labels = ["Output", "Query", "Key", "Value", "Pattern"]
    for c_idx, label in enumerate(component_labels):
        save_heatmap(
            avg_every[c_idx],
            f"Exp4 Average: {label} ({len(all_every_head_results)} sets)",
            f"exp4_head_{label.lower()}_AVERAGE.png",
        )

    # Cross-type comparison for top heads
    print("\n  Exp4 — Top heads cross-component comparison (averaged):")
    top_heads_from_exp2 = []
    for i in flat.abs().topk(min(6, flat.numel())).indices:
        layer = i.item() // n_heads
        head = i.item() % n_heads
        top_heads_from_exp2.append((layer, head))

    header = f"  {'Head':>8s}" + "".join([f"{l:>10s}" for l in component_labels])
    print(header)
    for l, h in top_heads_from_exp2:
        row = f"  L{l}H{h:>5d}"
        for c_idx in range(len(component_labels)):
            row += f"{avg_every[c_idx, l, h].item():+10.4f}"
        print(row)

    # Save tensors for later analysis
    torch.save(avg_attn, OUTPUT_DIR / "exp2_attn_head_AVERAGE.pt")
    torch.save(avg_every, OUTPUT_DIR / "exp4_every_head_AVERAGE.pt")
    print(f"\n  Tensors saved to {OUTPUT_DIR}/")

print("\n✓ Pipeline complete.")


Aggregate results over 5 sets
  Saved: patching_results/exp2_attn_head_AVERAGE.png

  Exp2 — Top 10 heads (averaged):
    #1: L17H2 = +1.0657
    #2: L16H0 = -1.0042
    #3: L13H2 = +0.7357
    #4: L14H1 = +0.6488
    #5: L17H1 = -0.6243
    #6: L9H1 = +0.4957
    #7: L14H0 = +0.4231
    #8: L16H1 = +0.4041
    #9: L16H3 = +0.2643
    #10: L11H1 = +0.2458
  Saved: patching_results/exp4_head_output_AVERAGE.png
  Saved: patching_results/exp4_head_query_AVERAGE.png
  Saved: patching_results/exp4_head_key_AVERAGE.png
  Saved: patching_results/exp4_head_value_AVERAGE.png
  Saved: patching_results/exp4_head_pattern_AVERAGE.png

  Exp4 — Top heads cross-component comparison (averaged):
      Head    Output     Query       Key     Value   Pattern
  L17H    2   +1.0657   +0.7385   +0.0000   +0.0000   +0.6037
  L16H    0   -1.0042   -0.7813   -0.3784   +0.0022   -0.4066
  L13H    2   +0.7357   +0.3266   +0.0000   +0.0000   +0.3374
  L14H    1   +0.6488   +0.1516   +0.0000   +0.0000   +0.0436
  